<picture>
  <source media="(prefers-color-scheme: dark)" srcset="https://raw.githubusercontent.com/thinkube/meepsquared/main/assets/meepsquared-logo-dark.svg">
  <img alt="Meep²" src="https://raw.githubusercontent.com/thinkube/meepsquared/main/assets/meepsquared-logo-light.svg" width="239" height="80">
</picture>

# 03 · LoRA on every expert of a mixture of experts

> **Licence: Apache-2.0.** Copyright Alejandro Martínez Corriá and the Thinkube contributors.
> This notebook, [meepsquared](https://github.com/thinkube/meepsquared) and every package meepsquared installs are permissively licensed:
> you can use, change and ship what you build with them, commercially or not. See [Licensing](https://thinkube.org/meepsquared/licensing.html).

**Where it runs.** On any Linux machine with an NVIDIA GPU of the Ampere generation or later and Meep² installed ([Install](https://github.com/thinkube/meepsquared#install)). The model and the data come from the Hugging Face Hub. Run the cells from top to bottom.

A few cells are marked **Thinkube only**. [Thinkube](https://thinkube.org) is the open-source AI platform Meep² is developed on; on a Thinkube cluster these cells read the model from the cluster's copy, record the run and serve the result. Elsewhere they do nothing, so run every cell as it comes.

The outputs saved in this notebook come from a run on a Thinkube cluster, on an NVIDIA DGX Spark, so they show the cluster's model paths and its run links.

## What you will do

Fine-tune a mixture-of-experts model with LoRA on its attention, its shared expert and **every routed expert**, with a training job: `m2.TrainingJob` and `m2.Recipe(experts=True)`. Notebooks [01](01-lora-sft.ipynb) and [02](02-fp8-weights.ipynb) explain LoRA and FP8; this one explains what is different about experts.

### What a mixture of experts is

A dense model runs every token through the same MLP in every layer. A **mixture of experts** (MoE) replaces that MLP with many smaller ones, the *experts*, and a small **router** that picks a few of them (8 of 256 here) for each token. Qwen3.6-35B-A3B has 35 billion parameters but *activates* about 3 billion per token: the quality of a large model at the compute of a small one. A *shared expert* runs on every token besides the routed ones.

### Why the experts are hard to fine-tune

In Transformers, the experts of a layer are not 256 separate `Linear` modules but one **stack** of weights per projection, a 3-D tensor. PEFT's `target_modules` names modules, so it cannot reach them; PEFT's `target_parameters` can, by naming the parameter. The question is then how to compute each expert's adapter on the tokens routed to it without a loop of 256 small products per layer, per step. meepsquared does it in one kernel launch per product: a grouped kernel that walks the experts and the tokens routed to each, with the adapter fused in. The expert stacks, most of the model's weights, can also be held as FP8 (`FP8 = True`): on an NVIDIA GB10, Qwen3.6-35B-A3B then trains at 857 training tokens per second in 61 GiB, against 799 in 79 GiB with BF16 stacks and 587 with PEFT's own computation ([benchmarks](https://thinkube.org/meepsquared/benchmarks.html)).

### What this needs

Qwen3.6-35B-A3B is 70 GB in BF16: a GPU of 96 GB or more, or 64 GB with `FP8 = True`. On Thinkube's DGX Spark it loads in about a minute from the cluster's mirror, never held twice in memory. The batch is larger than in the dense notebooks (32 conversations) because the experts work best with many tokens per step: each sees only its share.

Needs `meepsquared` and `pandas`; on Thinkube, the `meepsquared` kernel has them.

## Settings

Every choice of the notebook is a name in this cell, so you can rerun the notebook with other values without touching the cells below, and a notebook job can override any of them with its parameters.

In [2]:
PARAMETERS = globals().get("PARAMETERS", {})

MODEL_SOURCE = PARAMETERS.get("MODEL_SOURCE", "huggingface")      # "huggingface"; "thinkube" only on a Thinkube cluster
MODEL = PARAMETERS.get("MODEL", "Qwen/Qwen3.6-35B-A3B")
FP8 = PARAMETERS.get("FP8", True)                            # the frozen weights, the expert stacks included, held as FP8
DATASET = PARAMETERS.get("DATASET", "HuggingFaceH4/no_robots")
ROWS = PARAMETERS.get("ROWS", 960)
EPOCHS = PARAMETERS.get("EPOCHS", None)
STEPS = PARAMETERS.get("STEPS", 30)
BATCH = PARAMETERS.get("BATCH", 32)                          # conversations per step; the experts like large batches
MAX_LENGTH = PARAMETERS.get("MAX_LENGTH", 1024)
LEARNING_RATE = PARAMETERS.get("LEARNING_RATE", 1e-4)
RANK = PARAMETERS.get("RANK", 16)
OUTPUT_DIR = PARAMETERS.get("OUTPUT_DIR", "out/03-moe-experts")
# Thinkube only: ignored elsewhere
EXPERIMENT = PARAMETERS.get("EXPERIMENT", "meepsquared")
RUN_NAME = PARAMETERS.get("RUN_NAME", f"03-moe-{MODEL.split('/')[-1]}")

## Where the model comes from

A fine-tuning starts from a **base model**: a model somebody already trained on a great deal of text, whose behaviour you will adjust with your own data. The base model is the largest thing a training handles, 8 to 70 GB of weights for the models in these notebooks, so where it comes from matters.

`MODEL_SOURCE` chooses:

* `"huggingface"`: the model is downloaded from the Hugging Face Hub by its id, once, into the Hub's cache on this machine. A *gated* model, such as Llama, needs its licence accepted on its Hub page and the token of your account in the `HF_TOKEN` variable.
* `"thinkube"`, only on a [Thinkube](https://thinkube.org) cluster: the model comes from the cluster's **model mirror**: a copy made once into the cluster's shared storage, which every GPU node reads. Nothing is downloaded by a training, a gated model needs no token here, and a 70 GB model loads in about a minute. A model is mirrored in Thinkube Control (AI > Models). `thinkube_models.mirrored_model_path` finds its folder, and stops with an error naming the model if it is not mirrored, before any time is spent.

In [3]:
import os
from pathlib import Path

if MODEL_SOURCE == "thinkube":
    import thinkube_models as tkm

    MODEL_PATH = tkm.mirrored_model_path(MODEL)
    print(f"{MODEL}: {MODEL_PATH}")
elif MODEL_SOURCE == "huggingface":
    MODEL_PATH = MODEL
    print(f"{MODEL}: from the Hugging Face Hub")
else:
    raise ValueError(f"MODEL_SOURCE must be 'thinkube' or 'huggingface', not {MODEL_SOURCE!r}")

Qwen/Qwen3.6-35B-A3B: /home/thinkube/thinkube/mlflow/artifacts/1/8ff34d7aca184a4d83f10ba5403a460a/artifacts/model


## The data

Supervised fine-tuning learns from examples of the behaviour you want. Here the examples are conversations. Each one is a list of turns, `{"role": "user" | "assistant", "content": ...}`, the shape the model's **chat template** turns into the exact text the model sees: special tokens mark where each turn begins and ends, so the model learns where an answer starts. Any dataset in that shape works:

* a Hugging Face dataset id with a `messages` column;
* a Parquet file, or a JSONL file with one conversation per line, with the same `messages` field.

`ROWS` takes the first conversations of the dataset, `None` all of them. The default, `HuggingFaceH4/no_robots`, is 10,000 conversations written by people: varied, clean and small, a good first dataset. A real fine-tune uses your own conversations. A few hundred good ones already change a model's tone; a few thousand change what it knows how to do.

The cell prints the first conversation, so you can see the shape the model will learn from. The training job below takes this dataset as it is.

In [4]:
import json

from datasets import Dataset, load_dataset


def conversations(source, rows=None):
    """Conversations as {"messages": [...]}, from a Hub dataset, a Parquet file or a JSONL file."""
    if source.endswith(".jsonl"):
        data = Dataset.from_list([json.loads(line) for line in open(source) if line.strip()])
    elif source.endswith(".parquet"):
        data = load_dataset("parquet", data_files=source, split="train")
    else:
        data = load_dataset(source, split="train")
    if "messages" not in data.column_names:
        raise ValueError(f"{source} has no 'messages' column; its columns are {data.column_names}")
    if rows is not None:
        if len(data) < rows:
            raise ValueError(f"{source} has {len(data)} conversations; ROWS asks for {rows}")
        data = data.select(range(rows))
    return data.map(lambda r: {"messages": [{"role": m["role"], "content": m["content"]} for m in r["messages"]]},
                    remove_columns=data.column_names)


train_data = conversations(DATASET, ROWS)
print(f"{len(train_data)} conversations; the first:")
for turn in train_data[0]["messages"]:
    print(f"  {turn['role']}: {turn['content'][:200]!r}")

README.md:   0%|          | 0.00/5.61k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  571kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

960 conversations; the first:
  user: 'Please summarize the goals for scientists in this text:\n\nWithin three days, the intertwined cup nest of grasses was complete, featuring a canopy of overhanging grasses to conceal it. And decades later'
  assistant: 'Scientists are studying nests hoping to learn about transitional habitats that could help restore the shoreline of San Francisco Bay.'


## Thinkube only: the training run in Thinkube Experiments

*Elsewhere the cell only sets `REPORT_TO` to an empty list; run it anyway, the training job below uses it.*

A training is an experiment: settings in, a loss curve and a model out. Keeping that record is what lets you compare two runs later, or trace a served model back to the data and settings that made it. On Thinkube the record is **Thinkube Experiments** (MLflow). The cell opens a *run* there, before the training job is described: the job's `report_to=REPORT_TO` then sends every step's loss to that run as it trains, so the curve is on the experiments page while the run goes. Elsewhere the metrics are in the trainer's log below, and `REPORT_TO` is empty.

In [5]:
RUN_ID = None
if MODEL_SOURCE == "thinkube":
    import mlflow
    import thinkube_models as tkm

    os.environ["MLFLOW_TRACKING_TOKEN"] = tkm.get_mlflow_token()
    mlflow.set_tracking_uri("http://mlflow.mlflow.svc.cluster.local:5000")
    experiment = mlflow.get_experiment_by_name(EXPERIMENT)
    experiment_id = experiment.experiment_id if experiment else mlflow.create_experiment(EXPERIMENT)
    run = mlflow.start_run(experiment_id=experiment_id, run_name=RUN_NAME)
    RUN_ID = run.info.run_id
    REPORT_TO = ["mlflow"]
    print(f"run {RUN_ID} in experiment {EXPERIMENT!r}: open Thinkube Experiments to follow it")
else:
    REPORT_TO = []

run 36744f7f99394f838cf93bb73c41b183 in experiment 'meepsquared': open Thinkube Experiments to follow it


## The training job and its plan

`m2.TrainingJob` describes the training: the model, the data, the method and how long. `FP8` chooses the method: `"fp8"` holds the frozen weights, the expert stacks included, as FP8 once the adapters are added; `"lora"` keeps them in BF16. The routers, small and sensitive, stay in BF16 either way.

Two fields of the recipe (`m2.Recipe`, the training choices) say where the adapters go:

* `targets` names the dense projections: the attention, the gated DeltaNet projections of the linear-attention layers (Qwen 3.6 alternates two kinds of attention), and the shared expert.
* `experts=True` adds the two expert stacks of every layer, `gate_up_proj` and `down_proj`. meepsquared finds them in the model and gives their names to PEFT as `target_parameters`.

The recipe also sets the batch: `BATCH` conversations per update of the weights, 32 by default, because the experts work best with many tokens per step: each sees only its share. It sets the learning rate, and a cosine schedule: large steps first and small ones at the end, after a short warm-up. Auto mode chooses the rest from what it measures.

`job.plan()` does the work that comes before training. It loads the model, adds LoRA, measures the memory one batch needs on your GPU, and builds TRL's trainer. Nothing is trained yet. Two lines of the plan matter here:

* `experts` names the expert stacks and says which code computes them on this GPU. On GPUs where PyTorch's grouped product is a loop over the experts (the GB10, the RTX 30, 40 and 50 series), meepsquared's grouped kernel computes all experts in one launch per product.
* `checkpoint` says how many layers recompute their forward in the backward pass to fit the memory. One layer's activations for 32 × 1024 tokens are several GiB, because the routed tokens are gathered per expert, so most layers are checkpointed even on a 122 GiB GPU.

The count printed at the end is large, about 940 M, because an adapter of rank 16 on 256 experts in 40 layers is still 256 × 40 pairs of matrices. It is 2.7% of the model.

In [6]:
import meepsquared as m2

job = m2.TrainingJob(
    model=MODEL_PATH, data=train_data, method="fp8" if FP8 else "lora", output=OUTPUT_DIR,
    steps=None if EPOCHS else STEPS, epochs=EPOCHS, report_to=REPORT_TO,
    recipe=m2.Recipe(
        rank=RANK, learning_rate=LEARNING_RATE, effective_batch=BATCH, max_length=MAX_LENGTH, schedule="cosine",
        targets=["q_proj", "k_proj", "v_proj", "o_proj",       # full attention
                 "in_proj_qkv", "in_proj_z", "out_proj",       # gated DeltaNet
                 "gate_proj", "up_proj", "down_proj"],         # shared expert
        experts=True),                                         # every routed expert
)
print(job.plan())  # loads the model, adds LoRA, measures the memory; nothing is trained yet

trainer = job.trainer()  # TRL's SFTTrainer, set up as job.train() will run it
model, tokenizer = trainer.model, trainer.processing_class
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{trainable / 1e6:.0f} M trainable parameters")

Loading weights:   0%|          | 0/693 [00:00<?, ?it/s]

╭────────────╮      Meep² 0.1.0: fast LoRA and QLoRA on Transformers, PEFT and TRL
│ meep meep! │      GPU: NVIDIA GB10 (sm_121), allocator: expandable segments. PyTorch 2.11.0a0. Transformers 5.20.2. TRL 1.16.0
╰──┬─────────╯      Model: qwen3_5_moe_text, FP8 e4m3 frozen weights computing in bfloat16. Liger: on. Padding-free: on
   ╵ ∘ ○  ·  ·  ·   Apache-2.0: https://github.com/thinkube/meepsquared


[RANK 0] You are using a per_device_train_batch_size of 1 with padding-free training. Using a batch size of 1 annihilate the benefits of padding-free training. Please consider increasing the batch size to at least 2.


Tokenizing train dataset:   0%|          | 0/960 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/960 [00:00<?, ? examples/s]

/home/thinkube/venvs/custom/meepsquared/arm64/lib/python3.12/site-packages/meepsquared/_trl.py:258: FutureWarning: `use_liger_kernel=True` is deprecated and will be removed in v2.0.0. Use the Hub kernels instead, with `model_init_kwargs={"use_kernels": True}`.
  trainer_init(*bound.args, **bound.kwargs)


Cutting to max_length:   0%|          | 0/960 [00:00<?, ? examples/s]

/home/thinkube/venvs/custom/meepsquared/arm64/lib/python3.12/site-packages/meepsquared/_trl.py:258: FutureWarning: `use_liger_kernel=True` is deprecated and will be removed in v2.0.0. Use the Hub kernels instead, with `model_init_kwargs={"use_kernels": True}`.
  trainer_init(*bound.args, **bound.kwargs)


meepsquared plan for /home/thinkube/thinkube/mlflow/artifacts/1/8ff34d7aca184a4d83f10ba5403a460a/artifacts/model, LoRA on FP8 weights, on NVIDIA GB10 (65 GiB free)
  loading      shard by shard to GPU 0 (the GPU shares the system's memory)
  allocator cap 101.9 GiB (the system's available memory less 4 GiB for the page cache and the machine's daemons; an overflow is this process's error, not the machine's)
  weights      FP8 e4m3, computing in bfloat16 (method fp8)
  attention    flash_attention_2 (FlashAttention 2 with packed sequences)
  batches      packed, no padding (the attention (flash_attention_2) keeps the examples apart)
  Liger        kernels and fused loss, without its SwiGLU: the fused LoRA MLP runs instead (from_pretrained's default)
  fused LoRA   on (the model's MLP has a fused kernel)
  compile      layers not compiled (FlashAttention runs outside them)
  checkpoint   26 of 40 layers (the timed steps peaked at 95.68 GiB, 12.06 GiB over the budget of 83.63 GiB: 26 of 40

## Before training: what the base model answers

Before changing anything, ask the model two questions and keep its answers. The same questions come back after training. The loss curve will tell you that the training worked on the data; the answers tell you what it did to the model's behaviour, which is what you care about.

Two details of the helper are worth knowing. `for_inference` puts the model in generation mode (gradient checkpointing off, the layers uncompiled, the key-value cache on) and `for_training` puts it back as the plan set it, the checkpointed and compiled layers included; `answer` does both around one generation, so you can call it any time during the notebook, between the plan and the training too. And `enable_thinking=False` asks a thinking model (Qwen 3) for a direct answer; a chat template without that switch ignores it.

Generation runs through the same routed computation as training, so a sample costs nothing to set up.

In [7]:
import torch


def answer(model, tokenizer, question, max_new_tokens=160):
    """The model's answer to one user turn, with generation set up and training set back."""
    m2.MeepLanguageModel.for_inference(model)
    try:
        # enable_thinking=False keeps a thinking model's answer short; a template without the switch ignores it
        prompt = tokenizer.apply_chat_template([{"role": "user", "content": question}], tokenize=False,
                                               add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    finally:
        m2.MeepLanguageModel.for_training(model)


QUESTIONS = ["Explain in two sentences why the sky is blue.",
             "Write a haiku about a lighthouse keeper."]

before = {q: answer(model, tokenizer, q) for q in QUESTIONS}
for q, a in before.items():
    print(f"Q: {q}\nA: {a}\n")

Q: Explain in two sentences why the sky is blue.
A: Sunlight reaches Earth's atmosphere and is scattered in all directions by all the gases and particles in the air. Blue light is scattered more than other colors because it travels in shorter, smaller waves, which is why we see a blue sky most of the time.

Q: Write a haiku about a lighthouse keeper.
A: Salt on weathered hands,
Guiding ships through stormy night,
Light cuts through the dark.



## Train

`job.train()` prints the plan again, then trains with TRL's `SFTTrainer` and every setting of the plan: the batch, split into micro-batches that fit the memory and added up by gradient accumulation; the learning rate and its schedule; the 8-bit AdamW optimizer, which keeps the optimizer state small; padding-free batches, Liger's fused loss, and the checkpointed layers it measured. It returns a result: the steps, the time, the last loss and the peak memory.

The settings that matter most, all in the settings cell:

* `LEARNING_RATE`: how far each step moves the adapters. `1e-4` is a usual value for LoRA; `2e-4` learns faster and risks instability, `5e-5` is careful.
* `BATCH`: conversations per update of the weights. Larger is steadier and, here, keeps every expert busy.
* `STEPS` or `EPOCHS`: how long. The defaults run a fixed number of steps to show the mechanics; a real fine-tune sets `EPOCHS` and goes through its data one to three times.

### Counting the loss on the answers only

A conversation has two kinds of tokens: the user's words and the assistant's. The model should learn to *write* the assistant's turns, not to predict the user's, so the job counts the loss on the assistant turns only. It does that by default for conversations. The plan's `loss on` line says how: with TRL's own assistant-only loss where TRL can mask the model's chat template, else with meepsquared's `train_on_responses_only` and the turn markers it reads from the chat template. `m2.Recipe(answers_only=False)` counts every token instead.

In [8]:
result = job.train()
print(f"{result.steps} steps in {result.seconds:.0f} s; final loss {result.final_loss:.3f}; "
      f"peak memory {result.peak_memory_bytes / 2**30:.1f} GiB")

meepsquared plan for /home/thinkube/thinkube/mlflow/artifacts/1/8ff34d7aca184a4d83f10ba5403a460a/artifacts/model, LoRA on FP8 weights, on NVIDIA GB10 (65 GiB free)
  loading      shard by shard to GPU 0 (the GPU shares the system's memory)
  allocator cap 101.9 GiB (the system's available memory less 4 GiB for the page cache and the machine's daemons; an overflow is this process's error, not the machine's)
  weights      FP8 e4m3, computing in bfloat16 (method fp8)
  attention    flash_attention_2 (FlashAttention 2 with packed sequences)
  batches      packed, no padding (the attention (flash_attention_2) keeps the examples apart)
  Liger        kernels and fused loss, without its SwiGLU: the fused LoRA MLP runs instead (from_pretrained's default)
  fused LoRA   on (the model's MLP has a fused kernel)
  compile      layers not compiled (FlashAttention runs outside them)
  checkpoint   26 of 40 layers (the timed steps peaked at 95.68 GiB, 12.06 GiB over the budget of 83.63 GiB: 26 of 40

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Step,Training Loss


30 steps in 466 s; final loss 1.904; peak memory 74.6 GiB


### Reading the loss

The loss measures how surprised the model is by each next token of the training answers, averaged over the step: lower means it predicts the answers better. It falls quickly in the first steps, while the model learns the format and tone of the data, and more slowly after. A loss that stops falling says the model has learned what this data teaches at this learning rate; a loss that climbs says the learning rate is too high. Thirty steps show the first fall; a real fine-tune runs epochs over the data and watches the loss on held-out conversations too.

In [9]:
import pandas as pd

log = pd.DataFrame([h for h in trainer.state.log_history if "loss" in h])[["step", "loss"]]
display(log.set_index("step").T if len(log) <= 40 else log.set_index("step").iloc[::max(1, len(log) // 40)].T)

step,1,2,3,4,5,6,7,8,9,10,...,21,22,23,24,25,26,27,28,29,30
loss,1.718891,1.853196,1.716833,1.799096,1.816378,1.634585,1.839546,1.724358,1.654392,1.608521,...,1.849313,1.627407,1.672895,1.689271,1.59546,1.742296,1.626998,1.784564,1.755681,1.903658


## After training: the same questions

The answers move toward the style of the training data. no_robots is written by people answering plainly, so even a short run tends to make the model more direct and less list-like. After a handful of steps the change is small, and it may be none for a question the data says little about: the adapters' effect grows with the data and the steps. A few hundred steps on your own conversations give a model that writes like them.

In [10]:
for q in QUESTIONS:
    print(f"Q: {q}\nbefore: {before[q]}\nafter:  {answer(model, tokenizer, q)}\n")

Q: Explain in two sentences why the sky is blue.
before: Sunlight reaches Earth's atmosphere and is scattered in all directions by all the gases and particles in the air. Blue light is scattered more than other colors because it travels in shorter, smaller waves, which is why we see a blue sky most of the time.
after:  The sky is blue because of the way the atmosphere scatters sunlight. The atmosphere is made up of tiny particles that scatter sunlight in all directions, but blue light is scattered more than other colors because it travels in shorter, smaller waves. This is why we see a blue sky during the day.

Q: Write a haiku about a lighthouse keeper.
before: Salt on weathered hands,
Guiding ships through stormy night,
Light cuts through the dark.
after:  Lighthouse keeper
Lights the lamp to guide ships home
Safe in the night



## Save

Only the adapter is saved here. A merged save of adapters on expert stacks is not offered, so the model is served as base model plus adapter, which vLLM and Transformers load with PEFT. The adapter of every expert of 40 layers is about 2 GB, far more than a dense model's, and still a small fraction of the 70 GB base.

In [11]:
result.save(f"{OUTPUT_DIR}/adapter")
print(f"adapter saved to {OUTPUT_DIR}/adapter")

if RUN_ID:
    mlflow.end_run()

adapter saved to out/03-moe-experts/adapter
🏃 View run 03-moe-Qwen3.6-35B-A3B at: http://mlflow.mlflow.svc.cluster.local:5000/#/experiments/6/runs/36744f7f99394f838cf93bb73c41b183
🧪 View experiment at: http://mlflow.mlflow.svc.cluster.local:5000/#/experiments/6


## Where to go next

* [04 · GRPO](04-grpo-zebra.ipynb): reinforcement learning with a reward a program computes.
* [Mixture of experts](https://thinkube.org/meepsquared/mixture-of-experts.html) in the documentation: the routed computation, the kernel, the FP8 stacks.